# Notebook para estudos de vector search   

### Vector Similatiry   
**Similatiry Metrics**   
* Cosine similarity    

#### Vecotr Search Strategies   
* k-nearest neighbors (KNN)    
</br>
* Approximated Nearest Neighbors (ANN)
  * Trade accuracy for speed gains
  * Examples of index algorihms:
    * Tree-based: ANNOY by Spotify
    * Proximity graphs: HNSW
    * Clustering: FAISS by Facebook
    * Hashing: LSH
    * Vector compression: SCaNN by Google, Product Quantization (PQ)

#### Detalhamento da LSH   

O Problema: Busca de Similaridade em Grande Escala
Em bases de dados com milhões de itens, como textos, imagens ou produtos, encontrar itens semelhantes a um item de busca usando métodos tradicionais, como a similaridade de cosseno, é inviável. O custo de tempo de computação é muito alto, pois a comparação precisa ser feita com cada item da base de dados.

A Solução: LSH (Hashing Sensível à Localidade)
A LSH é uma técnica que nos permite resolver esse problema de forma eficiente. Em vez de comparar todos os itens, a LSH agrupa os itens que são semelhantes em "baldes" (buckets). A lógica central é: se dois itens são semelhantes, eles têm uma alta probabilidade de serem hashados para o mesmo balde.

O Processo de Hashing com LSH
Representação dos Dados: O primeiro passo é converter cada item (um texto, por exemplo) em um vetor numérico, que é a sua representação em um espaço de dados.

Criação de Assinaturas (SimHash): Em seguida, usamos uma técnica como o SimHash para "compactar" cada vetor em uma assinatura, que é um código de bits (uma sequência de 0s e 1s). O SimHash garante que vetores semelhantes terão assinaturas com uma pequena distância de Hamming (ou seja, poucos bits diferentes).

Múltiplas Tabelas Hash: Para resolver o problema de que pequenas diferenças nas assinaturas podem fazer com que dois itens caiam em baldes diferentes, a LSH utiliza um sistema com múltiplas tabelas hash. A assinatura de cada item é dividida em pedaços, e cada pedaço é usado como a chave para uma tabela hash diferente.

O Funcionamento da Busca
Quando você quer encontrar itens semelhantes a um item de busca:

A LSH não percorre toda a base de dados.

Ela gera a assinatura do item de busca.

Usa os pedaços da assinatura para ir diretamente aos baldes correspondentes em cada uma das tabelas hash.

Somente os itens que estão nesses baldes são candidatos a serem retornados. A partir daí, uma verificação mais precisa (como a similaridade de cosseno) pode ser feita, mas apenas com um número muito menor de itens.

#### Comparação de tempo de busca   

Comparando cosine similarity com e sem LSH.

In [0]:
#!pip install datasketch

In [0]:
import numpy as np
import time
from datasketch import MinHashLSH, MinHash
from multiprocessing import Pool, Manager

#### Criando os vetores para o estudo

In [0]:
# Cria uma função para gerar vetores semelhantes
def create_similar_vectors(base_vector, num_vectors=10, similarity_level=0.9):
    similar_vectors = []
    num_dimensions = len(base_vector)
    for _ in range(num_vectors):
        # Cria um vetor com a maioria dos elementos iguais ao vetor base
        similar_vector = np.copy(base_vector)
        # Inverte uma pequena porcentagem dos elementos para simular diferença
        num_changes = int(num_dimensions * (1 - similarity_level))
        change_indices = np.random.choice(num_dimensions, num_changes, replace=False)
        similar_vector[change_indices] = 1 - similar_vector[change_indices]
        similar_vectors.append(similar_vector)
    return similar_vectors

# Cria uma base de dados grande de vetores
data_size = 10000
vector_dim = 10000
vectors = []
for _ in range(data_size):
    # Vetores aleatórios (binários para simplificar)
    vectors.append(np.random.randint(2, size=vector_dim))

# Vamos adicionar um grupo de vetores muito semelhantes
base_vector_to_find = np.random.randint(2, size=vector_dim)
similar_group = create_similar_vectors(base_vector_to_find, num_vectors=20)
vectors.extend(similar_group)

### Indexação por LSH

In [0]:
# Função que será executada por cada processo
def create_minhashes_task(task_data):
    """
    Cria MinHash para um pedaço dos vetores.
    Retorna uma lista de tuplas (índice, MinHash).
    """
    start_index, chunk, num_perm = task_data
    minhashes_list = []
    
    # Cria a MinHash para cada vetor no pedaço (chunk)
    for i, vector in enumerate(chunk):
        m = MinHash(num_perm=num_perm)
        for val_i, val in enumerate(vector):
            if val == 1:
                m.update(str(val_i).encode('utf8'))
        # Retorna o índice global e o objeto MinHash
        minhashes_list.append((start_index + i, m))
        
    return minhashes_list

# ---

# PARALELISMO COM MULTIPROCESSING
# Configurações para o paralelismo
num_processes = 6
num_perm = 128

print("Iniciando indexação LSH em paralelo...")
start_time_lsh_parallel = time.time()

# Divide os vetores em pedaços para cada processo
chunk_size = len(vectors) // num_processes
tasks = []
for i in range(num_processes):
    start_index = i * chunk_size
    # Pega o último pedaço para ter certeza que todos os vetores são incluídos
    if i == num_processes - 1:
        chunk = vectors[start_index:]
    else:
        chunk = vectors[start_index:start_index + chunk_size]
    # Adiciona a tarefa com o índice de início, o pedaço de vetores e o num_perm
    tasks.append((start_index, chunk, num_perm))

# Cria o pool de processos e executa as tarefas
with Pool(processes=num_processes) as pool:
    results = pool.map(create_minhashes_task, tasks)

# O processo principal coleta os resultados e insere no LSH
lsh = MinHashLSH(threshold=0.8, num_perm=num_perm)
all_minhashes = {}
for result_list in results:
    for i, m in result_list:
        key = f"vector_{i}"
        all_minhashes[key] = m
        lsh.insert(key, m)

end_time_lsh_parallel = time.time()
print(f"Tempo de indexação LSH em paralelo com {num_processes} processos: {end_time_lsh_parallel - start_time_lsh_parallel:.4f} segundos")

#

#### Similarity search com força bruta


In [0]:
def brute_force_search(query_vector, data, threshold=0.9):
    
    start_time = time.time()
    similar_items = []
    # Normalização dos vetores para o cálculo do cosseno
    query_norm = np.linalg.norm(query_vector)
    
    for item in data:
        item_norm = np.linalg.norm(item)
        
        if query_norm > 0 and item_norm > 0:
            similarity = np.dot(query_vector, item) / (query_norm * item_norm)
            if similarity >= threshold:
                similar_items.append(item)
                
    end_time = time.time()
    
    return similar_items, f"{end_time - start_time:.4f}"

# Nosso vetor de busca
fb = []
for n in range(20):
    query_vector = similar_group[n]

    brute_force_results, interval = brute_force_search(query_vector, vectors)
    fb.append(float(interval))

print(f"Tempo médio de busca por força bruta: {sum(fb) / 20:.4} segundos")


#### Similarity Search com LSH

In [0]:
query_minhash = MinHash(num_perm = num_perm)

lsh_ = []
for n in range(20):    
    query_vector = similar_group[0]
    for i, val in enumerate(query_vector):
        if val == 1:
            query_minhash.update(str(i).encode('utf8'))

    start_time_lsh = time.time()
    lsh_results_keys = lsh.query(query_minhash)
    end_time_lsh = time.time()
    lsh_.append(float(f"{end_time_lsh - start_time_lsh:.4f}"))  

print(f"Tempo médio de busca por força bruta: {sum(lsh_) / 20:.4} segundos")      

#### Comparação dos resultados.

In [0]:
0.2278 / 0.00001